# Prediction Notebook - Fragment-Based Molecular Generation
# フラグメントベース分子生成 予測ノートブック

HuggingFaceからダウンロードした学習済みモデルを使用して、任意のSMILESからフラグメントを抽出し、そのフラグメントを含む新しい分子を生成します。

### モデルの種類
- **T5Chem (RFFMG)**: T5ベースのTransformerモデル（`t5chem_copy`環境で実行）
- **SAFE**: SAFE-GPTモデル（GPT-2ベース）（`safe_copy`環境で実行）

### フラグメント化手法
- **BRICS**: BRICS分解による化学的に意味のあるフラグメント化
- **RC_CMS**: ランダムカットによるフラグメント化（SP3-SP3結合、環-環接続など）

## 0. 依存ライブラリのインストール

必要に応じてコメントを解除して実行してください。

In [ ]:
# 共通
# !pip install huggingface_hub rdkit pandas numpy

# T5Chem環境の場合
# !pip install t5chem==1.2.0 transformers==4.45.2

# SAFE環境の場合
# !pip install safe-mol==0.1.13 transformers==4.45.2

## 1. モデルのダウンロード

HuggingFace Hub (`sato-akinori/FFMG`) から学習済みモデルをダウンロードします。  
プライベートリポジトリの場合は、事前に `huggingface-cli login` でログインしてください。

In [ ]:
import os
import glob
import subprocess
from huggingface_hub import snapshot_download

# HuggingFaceからモデルをダウンロード
snapshot_download(
    repo_id='sato-akinori/FFMG',
    allow_patterns='models/*',
    local_dir='.'
)

# zipファイルを解凍
for zip_file in glob.glob('models/**/*.zip', recursive=True):
    subprocess.run(['unzip', '-o', zip_file, '-d', os.path.dirname(zip_file)], check=True)
    os.remove(zip_file)

print('モデルのダウンロードが完了しました。')

## 2. 設定

フラグメント化手法、生成パラメータなどを設定します。

In [ ]:
# ========================================
# 設定（必要に応じて変更してください）
# ========================================
FRAG_METHOD = 'brics'    # 'brics' or 'rc_cms'
N_SAMPLES   = 10         # 生成する分子数
NUM_BEAMS   = 10         # ビームサーチのビーム数
MAX_LENGTH  = 200        # 最大シーケンス長
RANDOM_SEED = 42         # 乱数シード

# モデルパス
T5CHEM_MODEL_PATH = f'models/t5chem/trained/rffmg/{FRAG_METHOD}/best_model/'
SAFE_MODEL_PATH   = f'models/safe_gpt/trained/safe/{FRAG_METHOD}/best_model'

## 3. ライブラリのインポートとヘルパー関数

In [ ]:
import sys
sys.path.insert(0, 'src')

import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw
from IPython.display import display

from func.fragmentation import (
    BRICSFragmentize,
    RandomFragmentize,
    PostProcessSelectFrags,
)


def fragmentize_smiles(smiles, frag_method='brics', ratio=0.6, big_ring_thres=7, seed=42):
    """
    SMILESをフラグメント化する。
    
    Returns:
        pass_frags: フラグメントSMILES（環上のR基をトリム済み、T5Chem用）
        pass_frags_np: フラグメントSMILES（未処理、SAFE用）
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f'Invalid SMILES: {smiles}')

    if frag_method == 'brics':
        frags = BRICSFragmentize(mol, returnSmiles=False)
        trim_r_on_ring = False
    elif frag_method == 'rc_cms':
        frags = RandomFragmentize(
            mol, returnSmiles=False,
            bigRingThres=big_ring_thres, rseed=seed, ratio=ratio
        )
        trim_r_on_ring = True
    else:
        raise ValueError(f'Unknown method: {frag_method}')

    if frags is None:
        return None, None

    pass_frags, pass_frags_np = PostProcessSelectFrags(
        frags,
        smallCarbonFilter=True,
        trimRgroupOnRing=trim_r_on_ring,
        uniquenize=(frag_method != 'brics'),
    )
    return pass_frags, pass_frags_np


def convert_dummy_atoms(smiles):
    """ダミーアトムの Isotope ラベルを除去する。"""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    for atom in mol.GetAtoms():
        if atom.GetAtomicNum() == 0:
            atom.SetIsotope(0)
    return Chem.MolToSmiles(mol)


print('ライブラリの読み込みが完了しました。')

## 4. 入力SMILESとフラグメント化

任意のSMILESを入力し、フラグメント化を行います。  
`input_smiles` を変更して異なる分子で試してみてください。

In [ ]:
# ========================================
# 入力SMILES（変更可能）
# ========================================
input_smiles = 'CC(C)Cc1ccc(C(C)C(=O)O)cc1'  # イブプロフェン

# SMILESの検証と正規化
mol = Chem.MolFromSmiles(input_smiles)
assert mol is not None, '無効なSMILESです。正しいSMILESを入力してください。'
canonical_smi = Chem.MolToSmiles(mol)
print(f'入力分子 (canonical SMILES): {canonical_smi}')

# フラグメント化
pass_frags, pass_frags_np = fragmentize_smiles(canonical_smi, frag_method=FRAG_METHOD)

if pass_frags is None:
    print('この分子はフラグメント化できません。別のSMILESまたは手法を試してください。')
else:
    print(f'\n--- フラグメント化結果 ---')
    print(f'手法: {FRAG_METHOD}')
    print(f'フラグメント (T5Chem用): {pass_frags}')
    print(f'フラグメント (SAFE用):   {convert_dummy_atoms(pass_frags_np)}')
    print(f'フラグメント数: {len(pass_frags.split("."))}')

    # 入力分子とフラグメントの描画
    frag_mols = [Chem.MolFromSmiles(f) for f in pass_frags.split('.')]
    frag_mols = [m for m in frag_mols if m is not None]
    if frag_mols:
        img = Draw.MolsToGridImage(
            [mol] + frag_mols,
            molsPerRow=4,
            subImgSize=(300, 300),
            legends=['入力分子'] + [f'Fragment {i+1}' for i in range(len(frag_mols))]
        )
        display(img)

---
## 5. T5Chemモデルによる分子生成

**注意**: このセクションは `t5chem_copy` 環境で実行してください。
```bash
conda activate t5chem_copy
```

In [ ]:
import subprocess
import tempfile

assert pass_frags is not None, 'フラグメント化が失敗しています。セクション4を確認してください。'

with tempfile.TemporaryDirectory() as tmpdir:
    # ソースファイル（フラグメント）の書き込み
    with open(os.path.join(tmpdir, 'test.source'), 'w') as f:
        f.write(pass_frags + '\n')

    # ターゲットファイル（ダミー、t5chemに必須）の書き込み
    with open(os.path.join(tmpdir, 'test.target'), 'w') as f:
        f.write(canonical_smi + '\n')

    pred_file = os.path.join(tmpdir, 'predictions.csv')

    cmd = [
        't5chem', 'predict',
        '--data_dir', tmpdir,
        '--model_dir', T5CHEM_MODEL_PATH,
        '--prediction', pred_file,
        '--num_beams', str(NUM_BEAMS),
        '--num_preds', str(N_SAMPLES),
        '--batch_size', '1',
    ]

    print(f'実行コマンド: {" ".join(cmd)}')
    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        print(f'エラー: {result.stderr}')
    else:
        t5chem_df = pd.read_csv(pred_file)
        print(f'\n--- T5Chem 生成結果 ---')
        print(f'入力フラグメント: {pass_frags}')
        display(t5chem_df)

        # 生成分子の描画
        pred_smiles = [
            t5chem_df.iloc[0, i]
            for i in range(1, min(N_SAMPLES + 1, len(t5chem_df.columns)))
            if isinstance(t5chem_df.iloc[0, i], str)
        ]
        pred_mols = [Chem.MolFromSmiles(s) for s in pred_smiles]
        pred_mols = [m for m in pred_mols if m is not None]
        if pred_mols:
            img = Draw.MolsToGridImage(
                pred_mols[:12], molsPerRow=4, subImgSize=(300, 300),
                legends=[Chem.MolToSmiles(m) for m in pred_mols[:12]]
            )
            display(img)

---
## 6. SAFEモデルによる分子生成

**注意**: このセクションは `safe_copy` 環境で実行してください。
```bash
conda activate safe_copy
```

In [ ]:
import safe
from safe.sample import SAFEDesign
from safe.tokenizer import SAFETokenizer
from safe.trainer.model import SAFEDoubleHeadsModel
import torch

assert pass_frags_np is not None, 'フラグメント化が失敗しています。セクション4を確認してください。'

# モデルの読み込み
safe_model = SAFEDoubleHeadsModel.from_pretrained(SAFE_MODEL_PATH)
safe_tokenizer = SAFETokenizer.from_pretrained(SAFE_MODEL_PATH)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
safe_model = safe_model.to(device)
safe_model.eval()
print(f'デバイス: {device}')

# SAFEDesign の作成
safe_generator = SAFEDesign(model=safe_model, tokenizer=safe_tokenizer)

In [ ]:
# フラグメントの準備（ダミーアトムの isotope を除去）
safe_input_frags = convert_dummy_atoms(pass_frags_np)
print(f'入力フラグメント: {safe_input_frags}')

# 生成
safe_generator.generation_config.diversity_penalty = 0.0

if len(safe_input_frags.split('.')) == 1:
    print('Scaffold Decoration モードで生成中...')
    gen_mols = safe_generator.scaffold_decoration(
        scaffold=safe_input_frags,
        n_samples_per_trial=N_SAMPLES,
        n_trials=1,
        do_sample=False,
        random_seed=RANDOM_SEED,
        max_length=MAX_LENGTH,
    )
else:
    print('Scaffold Morphing モードで生成中...')
    gen_mols = safe_generator.scaffold_morphing(
        side_chains=safe_input_frags.split('.'),
        n_samples_per_trial=N_SAMPLES,
        n_trials=1,
        random_seed=RANDOM_SEED,
        max_length=MAX_LENGTH,
    )

# SAFE → SMILES 変換
safe_predictions = []
for mol_safe in gen_mols:
    if mol_safe is None:
        safe_predictions.append(None)
        continue
    try:
        smi = safe.decode(mol_safe, canonical=True)
        safe_predictions.append(smi)
    except Exception:
        safe_predictions.append(None)

# 結果表示
valid_predictions = [s for s in safe_predictions if s is not None]
print(f'\n--- SAFE 生成結果 ---')
print(f'生成分子数: {len(gen_mols)}')
print(f'有効分子数: {len(valid_predictions)}')

safe_df = pd.DataFrame({
    'SAFE': gen_mols,
    'SMILES': safe_predictions,
})
display(safe_df)

# 生成分子の描画
pred_mols = [Chem.MolFromSmiles(s) for s in valid_predictions]
pred_mols = [m for m in pred_mols if m is not None]
if pred_mols:
    img = Draw.MolsToGridImage(
        pred_mols[:12], molsPerRow=4, subImgSize=(300, 300),
        legends=[Chem.MolToSmiles(m) for m in pred_mols[:12]]
    )
    display(img)